In [7]:
import os
import pandas as pd
from pandarallel import pandarallel
import numpy as np
import argparse
from tqdm import tqdm
import time
import unicodedata

In [9]:
import argparse
from tqdm import tqdm
import unicodedata

In [5]:
path = "C:/Users/sebratt/Box/Humility in Inquiry 2023\Data Sharing (Plos)/gender classification PLOS"
os.chdir(path)

In [8]:
df=pd.read_csv("C:/Users/sebratt/Box/Humility in Inquiry 2023\Data Sharing (Plos)/gender classification PLOS/plos_gender.csv")
df

,work_id,id,display_name,display_name_alternatives,country_code
0,https://openalex.org/W1512035175 ...,https://openalex.org/A5042773644,Vicente Planelles,"[""Vicente Planelles"", ""V. Planelles"", ""Vicente...",US
1,https://openalex.org/W1963589753 ...,https://openalex.org/A5069230203,Ming-Tzo Wei,"[""M. T. Wei"", ""Ming‐Tzo Wei"", ""Mingtzo Wei""]",US
2,https://openalex.org/W1963598789 ...,https://openalex.org/A5076276151,Peter D. Tonge,"[""Peter Tonge"", ""P. Tonge"", ""P.D. Tonge"", ""Pet...",CA
3,https://openalex.org/W1963604144 ...,https://openalex.org/A5062273084,William J. Brunken,"[""William J. Brunken"", ""W. J. Brunken"", ""W. Br...",US
4,https://openalex.org/W1963604144 ...,https://openalex.org/A5062273084,William J. Brunken,"[""William J. Brunken"", ""W. J. Brunken"", ""W. Br...",US
...,...,...,...,...,...
139423,https://openalex.org/W2523805925 ...,https://openalex.org/A5064255152,Jorge Luis Parra Arango,"[""Juan David Arango"", ""Jesús Arango"", ""Juan D....",CO
139424,https://openalex.org/W1983824430 ...,https://openalex.org/A5041864474,Koji Okihara,"[""K. Obata"", ""Obata Koji"", ""Koji Okudela"", ""Ok...",JP
139425,https://openalex.org/W2025611203 ...,https://openalex.org/A5064871460,Masahiro Yokouchi,"[""M. Yokouchi"", ""Yokouchi Masahiro"", ""Masahiro...",JP
139426,https://openalex.org/W2114674005 ...,https://openalex.org/A5051462519,Qiuyun Fan,"[""Q. Fan"", ""Qiuyun Fan""]",GB


In [10]:
'''
Author: Sarah Stueve
Last updated: 11/3/2025

Description: This script will process names from OpenAlex in order to split them into given name and 
last name.

PACKAGES REQUIRED:
- pandas
- argparse
- tdqm
- unicodedata
'''

# imports
import pandas as pd
import argparse
from tqdm import tqdm
import unicodedata

# function definitions
def remove_invalid_accents(input_str):
    '''
    This function normalizes an input string (or name) into NFKD form and encodes the string
    in UTF-8 to avoid invalid UTF-8 errors
    Parameters:
        :input_str: str, author name
    Returns:
        updated input_str with correct encoding.
    '''
    if pd.notna(input_str):
        nfkd_form = unicodedata.normalize('NFKD', input_str)
        # print(nfkd_form)
        utf8_encoded = nfkd_form.encode(errors = 'replace')
        # print(utf8_encoded)
        return utf8_encoded.decode()        
    return input_str

def contains_control_characters(s):
    '''
    This function returns True if there are control characters in string "s", False otherwise
    Parameters:
        :s: str, a string
    Returns:
        bool, True or False
    '''
    for char in s:
        if unicodedata.category(char) == 'Cc':  # 'Cc' is the category for control characters
            return True
    return False

def is_valid_utf8(data: bytes) -> bool:
    """
    Checks if a given byte string is valid UTF-8.

    Args:
        data: The byte string to check.

    Returns:
        True if the byte string is valid UTF-8, False otherwise.
    """
    # data = data.encode('utf-8', errors = 'replace')
    try:
        data.decode('utf-8')
        return True
    except UnicodeDecodeError:
        return False

def process_display_name(name):
    '''
    This function takes a name from OpenAlex (either from display_name field or display_name_alternatives) and returns that name 
    split into my best estimation of the author's given name and last name. It assumes you have already checked for "." characters 
    and commas/any other characters to avoid.

    Parameters:
        :name: str, an instance of an OpenAlex display name
    Returns:
        A tuple of given_name, last_name
    '''
    # check to see if "del", "de la", "y", "Van", "Von" etc. in the name:
    substring_lst = [' de ', 
                    ' del ', 
                    ' de la ', 
                    ' y ', 
                    ' van ',
                    ' von ',
                    ' den ',
                    ' le ',
                    ' di ',
                    ' dos ']
    # remove accents
    name = remove_invalid_accents(name)
    present_substring = [substring for substring in substring_lst if substring in name.lower()]
    present_substring = present_substring[0] if present_substring else []
    if present_substring:
    # if so, find index of instance of character
        sub_idx = name.lower().find(present_substring)
        # first name = everything before that index
        given_name = name[:sub_idx].strip()
        # last name = everything from that index to end
        family_name = name[sub_idx:].strip()
    # otherwise:
    else:  
        # split on last instance of space 
        try:
            idx = name.rindex(' ')
        except:
            idx = None
            return None
        if idx:
            # first/given name = everything to left of space
            given_name = name[:idx].strip()
            # last/family name = everything to the right 
            # (+1 to start at the next character to the right of the space)
            family_name = name[idx + 1:].strip()
    return given_name.strip(), family_name.strip()

# initialize argparse
parser = argparse.ArgumentParser()
parser.add_argument('-f', '--fname', help='Input filename for name processing.')
parser.add_argument('-p', '--use_parquet', help='Pass True if the file is in parquet format.')
args = parser.parse_args()

# substring list for additional checks
substring_lst = ['de', 
                'del', 
                'de la', 
                'y', 
                'van',
                'von',
                'den',
                'le',
                'di',
                'dos']


In [17]:
# read in csv file for path passed in using argparse
fname = "C:/Users/sebratt/Box/Humility in Inquiry 2023/Data Sharing (Plos)/gender classification PLOS/plos_gender.csv"
survey_authors = pd.read_csv(args.fname)
survey_authors

AttributeError: 'Namespace' object has no attribute 'df'

In [ ]:
if eval(args.use_parquet):
    survey_authors = pd.read_parquet(args.fname)
else:
    survey_authors = pd.read_csv(args.fname)
# for each row (i.e., author),
for row in tqdm(survey_authors.index):
    # access author display name
    display_name = survey_authors.loc[row, 'display_name']
    # remove invalid accents
    display_name = remove_invalid_accents(display_name)
    # if the display_name isn't null and is valid utf-8
    if pd.notna(display_name) and is_valid_utf8(display_name.encode()):
        if '.' not in display_name and not contains_control_characters(display_name):
            name_processed = process_display_name(display_name)
            if name_processed:
                survey_authors.loc[row, 'given_name'], survey_authors.loc[row, 'family_name'] = name_processed[0], name_processed[1]
                continue
        # print(display_name)
        # rearrange name if , in display name
        if ',' in display_name and not contains_control_characters(display_name):
            split_name = display_name.split(",")
            display_name = split_name[1].strip() + ' ' + split_name[0].strip()
        split_display_name = display_name.split()
        # check to see if length of name is > 2 and there are multiple given names
        if (len(split_display_name) > 2) and (split_display_name[1].lower() not in substring_lst) and ((('.' in split_display_name[0]) and ('.' not in split_display_name[1])) 
            or (('.' in split_display_name[1]) and ('.' not in split_display_name[0])) and not contains_control_characters(display_name)):
            # process name if so
            name_processed = process_display_name(display_name)
            if name_processed:
                survey_authors.loc[row, 'given_name'], survey_authors.loc[row, 'family_name'] = name_processed[0], name_processed[1]
        # if there is more than one . or a period in the display name and the length of it is <= 2
        else:#(display_name.count('.') >= 1):# or (('.' in display_name) and (len(split_display_name) <= 2)): 
            for alt_name in eval(survey_authors.loc[row, 'display_name_alternatives']):
                if len(alt_name.split()) >= 2:
                    # check same condition for alt name AND first name first letter matches between alt_name and display_name:
                    if ',' in alt_name:
                        alt_name_split = alt_name.split(',')
                        alt_name = alt_name_split[1].strip() + ' ' + alt_name[0].strip()
                    alt_name_split = alt_name.split()
                    if '.' not in alt_name and is_valid_utf8(alt_name.encode()) and not contains_control_characters(alt_name):
                    # if (len(alt_name_split) > 2) and ((('.' in alt_name_split[0]) and ('.' not in alt_name_split[1])) 
                    #     or (('.' in alt_name_split[1]) and ('.' not in alt_name_split[0])) and (alt_name[0] == display_name[0])):
                        name_processed = process_display_name(alt_name)
                        if name_processed:
                            survey_authors.loc[row, 'given_name'], survey_authors.loc[row, 'family_name'] = name_processed[0], name_processed[1]
                            break # break inner loop to stop processing alt_names
        
print(survey_authors[['display_name', 'given_name', 'family_name']].head(10))
# print(survey_authors.columns)
if eval(args.use_parquet):
    survey_authors.to_parquet(args.fname.split('.')[0] + '_names_processed.parquet', index = False)
else:
    survey_authors.to_csv(args.fname.split('.')[0] + '_names_processed.csv', index = False)

